In [1]:
import sys
from pathlib import Path

notebook_dir = Path.cwd()
parent_dir = notebook_dir.parent

sys.path.append(str(parent_dir))

%load_ext autoreload
%autoreload 2

In [2]:
import boto3
from os import getenv
import json
from pprint import pprint
import awswrangler as wr
import pandas as pd
from sqlalchemy import text
from datetime import datetime, date
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np

from sqlalchemy import create_engine
import base64
from io import BytesIO

%matplotlib inline

In [3]:
session = boto3.Session(profile_name='SA', region_name='eu-central-1')

In [8]:
POSTHOG_QUERY= """SELECT up.*
FROM usage.posthog up 
WHERE up.event = 'dashboard_viewed';"""

In [9]:
posthog = wr.athena.read_sql_query(
            sql=POSTHOG_QUERY,
            database="usage",
            s3_output="s3://slimwonen-athena-queries/",
            workgroup="primary",
            boto3_session=session,
        )

In [11]:
posthog.head()

,uuid,timestamp,_inserted_at,created_at,event,properties,distinct_id,elements_chain,person_properties,person_id
0,019632fd-06e4-7af2-ae35-7bb515b94058,2025-04-14 06:29:25.398,2025-04-14 06:29:25.398,2025-04-14 06:29:37.660,dashboard_viewed,"{""$session_id"":""019632fc-fd01-7f48-bf62-8cb274...",0195ddcb-50a6-7153-bd8e-cf5b748d9039,,"{""$os"":""Android"",""$app_name"":""SlimWonen App"",""...",08583af1-dab4-5776-8e7b-7310de3fcf23
1,019633ae-49f9-7ec0-af94-39e840438b4a,2025-04-14 09:43:02.506,2025-04-14 09:43:02.506,2025-04-14 09:43:44.613,dashboard_viewed,"{""$active_feature_flags"":[],""$os_name"":""iOS"",""...",01933427-f8b2-7e39-bcf3-6443eafc1dab,,"{""$os"":""iOS"",""$app_name"":""SlimWonen App"",""$app...",08583af1-dab4-5776-8e7b-7310de3fcf23
2,019632d9-f9c1-7291-b0ea-dac2598a9ca8,2025-04-14 05:51:08.343,2025-04-14 05:51:08.343,2025-04-14 05:51:39.063,dashboard_viewed,"{""$app_version"":""2.4.2"",""$device_name"":""SM-A52...",0193ba30-b23f-70cf-9777-bfc89b59b904,,"{""$os"":""Android"",""$app_name"":""SlimWonen App"",""...",2ca3fc4e-faa8-56fb-816e-547d28418335
3,01963338-3b48-78dc-97f1-e89f2e180372,2025-04-14 07:34:03.276,2025-04-14 07:34:03.276,2025-04-14 07:36:46.235,dashboard_viewed,"{""$session_id"":""01963338-319c-767c-82a3-8d9ce4...",01935849-63ed-749f-839d-a28aaacd1bce,,"{""$os"":""Android"",""$app_name"":""SlimWonen App"",""...",79fb8133-b220-520b-a763-0e8116634f64
4,0196337d-2593-75f0-aedf-81c724132b3c,2025-04-14 08:49:21.849,2025-04-14 08:49:21.849,2025-04-14 08:50:39.979,dashboard_viewed,"{""$app_version"":""2.4.2"",""$screen_height"":844,""...",0193916f-25c1-7700-a9d9-0d33c18c70d8,,"{""$os"":""iOS"",""$app_name"":""SlimWonen App"",""$app...",04093871-f329-54ca-bb49-6759bbe3df16


In [13]:
def add_data(row: pd.Series) -> pd.Series:
    props = json.loads(row['properties'])
    if 'household_id' in props:
        row['household_id'] = props['household_id']
    return row

In [42]:
posthog.shape

(504792, 11)

In [15]:
posthog = posthog.apply(add_data, axis=1)

In [38]:
df = posthog[['household_id', 'person_id', 'distinct_id']]

In [17]:
# 1. How many unique household_ids per person_id or distinct_id?
df.groupby('person_id')['household_id'].nunique().describe()
# df.groupby('distinct_id')['household_id'].nunique().describe()

count    14301.000000
mean         1.264457
std         26.618499
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max       3184.000000
Name: household_id, dtype: float64

In [21]:
# 3. See the actual distribution (histogram of cardinality)
df.groupby('person_id')['household_id'].nunique().value_counts().sort_index()
# df.groupby('distinct_id')['household_id'].nunique().value_counts().sort_index()
# 1 → N mapping = broken; all 1s = usable

household_id
1       13845
2         404
3          26
4           9
5           7
6           1
7           2
8           1
11          1
12          1
13          1
15          1
18          1
3184        1
Name: count, dtype: int64

In [33]:
# 1. Find person_ids that map to exactly one household_id
person_household_counts = df.groupby('person_id')['household_id'].nunique()
clean_person_ids = person_household_counts[person_household_counts == 1].index

# 2. Build the mapping dict {person_id: household_id}
#    Use dropna() to exclude rows where household_id is already null
mapping = (
    df[df['person_id'].isin(clean_person_ids)]
    .dropna(subset=['household_id'])
    .groupby('person_id')['household_id']
    .first()  # safe since we know there's only one unique value
    .to_dict()
)

# 3. Apply: fill missing household_id using the mapping
df['household_id'] = df['household_id'].fillna(df['person_id'].map(mapping))

/var/folders/fr/dynq2jt113v40q17r07m78d40000gn/T/ipykernel_79252/3213183733.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['household_id'] = df['household_id'].fillna(df['person_id'].map(mapping))


In [43]:
posthog.sort_values(by='timestamp').head()

,uuid,timestamp,_inserted_at,created_at,event,properties,distinct_id,elements_chain,person_properties,person_id,household_id
225052,0192e81f-4e0a-7732-8ab7-df5248371ce8,2024-11-01 14:26:58.752,2024-11-01 14:26:58.752,2024-11-01 14:27:09.084,dashboard_viewed,"{""$screen_height"":667,""$device_type"":""Mobile"",...",0192e81c-064c-73e2-b454-0e33e021114f,,"{""$os"":""iOS"",""$app_name"":""slimwonen-v3"",""$app_...",84261b42-14d3-58d5-9134-5958f1b9212a,6c2fceb0-eb01-4d5a-a924-c918fc5742c5
225053,0192e820-ea94-75dc-9562-e831b31e11aa,2024-11-01 14:28:44.369,2024-11-01 14:28:44.369,2024-11-01 14:28:52.604,dashboard_viewed,"{""$os_name"":""iOS"",""$os_version"":""17.5"",""$devic...",0192e81c-064c-73e2-b454-0e33e021114f,,"{""$os"":""iOS"",""$app_name"":""slimwonen-v3"",""$app_...",84261b42-14d3-58d5-9134-5958f1b9212a,6c2fceb0-eb01-4d5a-a924-c918fc5742c5
225054,0192e831-4b59-7985-94c8-573783ba7dad,2024-11-01 14:46:37.723,2024-11-01 14:46:37.723,2024-11-01 14:46:46.265,dashboard_viewed,"{""$lib"":""posthog-react-native"",""$os_name"":""iOS...",0192e81c-064c-73e2-b454-0e33e021114f,,"{""$os"":""iOS"",""$app_name"":""slimwonen-v3"",""$app_...",84261b42-14d3-58d5-9134-5958f1b9212a,6c2fceb0-eb01-4d5a-a924-c918fc5742c5
242548,0192ed67-6ece-7472-89ff-d80aaab6f46f,2024-11-02 15:03:51.484,2024-11-02 15:03:51.484,2024-11-02 15:03:58.859,dashboard_viewed,"{""$screen_name"":""splash/index"",""$app_name"":""sl...",0192ed67-6358-7abb-9182-748197c60788,,"{""$os"":""HUAWEI/MAR-LX1AEEA/HWMAR:10/HUAWEIMAR-...",4b14aa45-fee2-5eab-9449-23cddb312c90,un_authenticated
264392,0192f0fe-f32b-7635-a60b-4f09a5ff0e6b,2024-11-03 07:48:13.274,2024-11-03 07:48:13.274,2024-11-03 07:48:21.518,dashboard_viewed,"{""$device_manufacturer"":""Apple"",""$screen_name""...",0192e81c-064c-73e2-b454-0e33e021114f,,"{""$os"":""iOS"",""$app_name"":""slimwonen-v3"",""$app_...",84261b42-14d3-58d5-9134-5958f1b9212a,un_authenticated


## Realised this was a pointless exercise since the household_id is also present in the oldest logs